# ` Intelligent E-Commerce Customer Analytics Platform `

# Step 1: Import Required Libraries

In [13]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
import seaborn as sns

# Visualization Settings
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

# Display all columns
pd.set_option("display.max_columns", None)

# Step 2: Load Dataset

In [6]:
df = pd.read_csv(
    "../data/processed"
    "/customer_analytics_cleaned.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

print("Dataset Loaded Successfully")

Dataset Loaded Successfully


# Session 1 - Customer Feature Engineering

In [7]:
# =====  Create Customer Feature Dataset ========

customer_features = pd.DataFrame()
customer_features["customer_unique_id"] = (
    df["customer_unique_id"].unique()
)
customer_features = customer_features.set_index(
    "customer_unique_id"
)

In [8]:
# ========= 1. Total Orders  ===========
customer_features["Total_Orders"] = (
    df.groupby("customer_unique_id")["order_id"]
      .nunique()
)


# ========= 2. Total Spending ==========
customer_features["Total_Spending"] = (
    df.groupby("customer_unique_id")["payment_value"]
      .sum()
)

# ========= 3. Average Order Value ========
customer_features["Average_Order_Value"] = (
    customer_features["Total_Spending"]  /
    customer_features["Total_Orders"]
)

# ========= 4. Basket Size =========
# Average number of products purchased per order.

basket_size = (df.groupby("customer_unique_id")
      .agg(
          Products=("product_id","count"),
          Orders=("order_id","nunique")
      )
)
customer_features["Average_Basket_Size"] = (
    basket_size["Products"] /
    basket_size["Orders"]
)

# ======= 5. Recency  =========
# How recently did the customer purchase?

snapshot_date = (df["order_purchase_timestamp"].max()  +  pd.Timedelta(days=1))
customer_features["Recency"] = (
    df.groupby("customer_unique_id")
      ["order_purchase_timestamp"].max().apply( lambda x:(snapshot_date - x).days )
)

# ======= 6. Frequency ======
# Measures purchase frequency.

customer_features["Frequency"] = (
    df.groupby("customer_unique_id")
      ["order_id"].nunique()
)

# ======= 7. Monetary Value (RFM) ==========
# Measures customer lifetime spending.

customer_features["Monetary"] = (
    df.groupby("customer_unique_id")
      ["payment_value"].sum()
)

# ======= 8. Customer Tenure ============
# Days between first purchase and latest purchase.

customer_tenure = (df.groupby("customer_unique_id")
                    .agg(
                        First=("order_purchase_timestamp","min"),
                        Last=("order_purchase_timestamp","max")
                        )
)
customer_features["Customer_Tenure"] = (customer_tenure["Last"] -  customer_tenure["First"]).dt.days


# ======== 9.  Days Since Last Purchase ==========
customer_features["Days_Since_Last_Purchase"] = (
    customer_features["Recency"]
)


# ======== 10.  Average Review Rating  ============
customer_features["Average_Review_Rating"] = (
    df.groupby("customer_unique_id")["review_score"].mean())


# ======== 11. Preferred Payment Method ================
preferred_payment = (
    df.groupby("customer_unique_id")["payment_type"]
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)
customer_features["Preferred_Payment_Method"] = preferred_payment

# =========== Feature Dataset Overview  ==============
customer_features.head()


,Total_Orders,Total_Spending,Average_Order_Value,Average_Basket_Size,Recency,Frequency,Monetary,Customer_Tenure,Days_Since_Last_Purchase,Average_Review_Rating,Preferred_Payment_Method
customer_unique_id,,,,,,,,,,,
861eff4711a542e4b93843c6dd7febb0,1,146.87,146.87,1.0,520,1,146.87,0,520,4.0,credit_card
290c77bc529b7ac935b93aa66c333dc3,1,335.48,335.48,1.0,278,1,335.48,0,278,5.0,credit_card
060e732b5b29e8181a18229c7b0b2b5e,1,157.73,157.73,1.0,152,1,157.73,0,152,5.0,credit_card
259dac757896d24d7702b9acbbff3f3c,1,173.30,173.30,1.0,219,1,173.30,0,219,5.0,credit_card
345ecd01c38d18a9036ed96c73b8d066,1,252.25,252.25,1.0,81,1,252.25,0,81,5.0,credit_card


In [9]:
customer_features.info()

<class 'pandas.DataFrame'>
Index: 96096 entries, 861eff4711a542e4b93843c6dd7febb0 to 84732c5050c01db9b23e19ba39899398
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Total_Orders              96096 non-null  int64  
 1   Total_Spending            96096 non-null  float64
 2   Average_Order_Value       96096 non-null  float64
 3   Average_Basket_Size       96096 non-null  float64
 4   Recency                   96096 non-null  int64  
 5   Frequency                 96096 non-null  int64  
 6   Monetary                  96096 non-null  float64
 7   Customer_Tenure           96096 non-null  int64  
 8   Days_Since_Last_Purchase  96096 non-null  int64  
 9   Average_Review_Rating     95380 non-null  float64
 10  Preferred_Payment_Method  96095 non-null  str    
dtypes: float64(5), int64(5), str(1)
memory usage: 12.6 MB


#  Missing Values 

In [12]:
# ======== Check Missing Values  =============
customer_features.isnull().sum()

Total_Orders                0
Total_Spending              0
Average_Order_Value         0
Average_Basket_Size         0
Recency                     0
Frequency                   0
Monetary                    0
Customer_Tenure             0
Days_Since_Last_Purchase    0
Average_Review_Rating       0
Preferred_Payment_Method    0
dtype: int64

In [11]:
# ======== Handling Missing Values  =============
customer_features["Average_Review_Rating"] = (
    customer_features["Average_Review_Rating"]
    .fillna(customer_features["Average_Review_Rating"].median())
)

customer_features["Preferred_Payment_Method"] = (
    customer_features["Preferred_Payment_Method"]
    .fillna("Unknown")
)

In [14]:
# ========= Save Engineered Features =================

# Output directory
output_dir = r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed"

# Output file path
output_path = os.path.join(output_dir, "customer_features.csv")

# Reset index before saving
customer_features.reset_index(inplace=True)

# Save CSV
customer_features.to_csv(output_path, index=False)

print(f"Customer feature dataset saved successfully!")
print(f"Location: {output_path}")

Customer feature dataset saved successfully!
Location: C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed\customer_features.csv


# Session 2 - Product Feature Engineering

In [20]:
# ====== Create Product Feature Dataset ============

product_features = pd.DataFrame()
product_features["product_id"] = (df["product_id"].unique())
product_features = product_features.set_index("product_id")

In [21]:
# ======== 1. Product Popularity ==============
product_features["Product_Popularity"] = (
    df.groupby("product_id")["order_id"].nunique())

# ======== 2. Average Product Rating ===========
product_features["Product_Rating"] = (
    df.groupby("product_id")["review_score"].mean()
)

# ======= 3. Category Popularity  ==========
category_popularity = (
    df.groupby("product_category_name")["order_id"].nunique()
)
product_features["Category_Popularity"] = (
    df.groupby("product_id")["product_category_name"].first()
      .map(category_popularity)
)

# ======= 4. Average Selling Price  =========
product_features["Average_Selling_Price"] = (
    df.groupby("product_id")["price"].mean()
)

product_features.head()

,Product_Popularity,Product_Rating,Category_Popularity,Average_Selling_Price
product_id,,,,
a9516a079e37a9c9c36b9b78b10169e8,40.0,3.393443,1273.0,119.285082
4aa6014eceb682077f9dc4bffebc05b0,45.0,4.361702,5884.0,291.160417
bd07b66896d6f1494f5b86251848ced7,8.0,3.800000,1273.0,143.958000
a5647c44af977b148e0a3a4751a09e2e,12.0,3.666667,1273.0,165.587200
9391a573abe00141c56e38d84d7d5b3b,4.0,3.250000,397.0,230.000000


# Missing Values

In [39]:
# ======== Check Missing Values  =============

print(product_features.info())
print(product_features.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   index                  32951 non-null  int64  
 1   product_id             32951 non-null  str    
 2   Product_Popularity     32951 non-null  float64
 3   Product_Rating         32951 non-null  float64
 4   Category_Popularity    32951 non-null  float64
 5   Average_Selling_Price  32951 non-null  float64
dtypes: float64(4), int64(1), str(1)
memory usage: 2.5 MB
None
index                    0
product_id               0
Product_Popularity       0
Product_Rating           0
Category_Popularity      0
Average_Selling_Price    0
dtype: int64


In [ ]:
# ======== Handling Missing Values  =============

product_features = product_features.dropna(subset=["product_id"]).reset_index(drop=True)
product_features["Product_Rating"] = (
    product_features["Product_Rating"]
    .fillna(product_features["Product_Rating"].median())
)
product_features["Category_Popularity"] = (
    product_features["Category_Popularity"]
    .fillna(product_features["Category_Popularity"].median())
)
product_features["Average_Selling_Price"] = (
    product_features["Average_Selling_Price"]
    .fillna(product_features["Average_Selling_Price"].median())
)

In [40]:
# ========= Save Engineered Features =================

# Output directory
output_dir = r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed"

# Output file path
output_path = os.path.join(output_dir, "product_features.csv")

# Reset index before saving
product_features.reset_index(inplace=True)

# Save CSV
product_features.to_csv(output_path, index=False)

print(f"Product feature dataset saved successfully!")
print(f"Location: {output_path}")

Product feature dataset saved successfully!
Location: C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed\product_features.csv


# Session 3 - Seller Feature Engineering

In [58]:
# ====== Create Seller Feature Dataset ============

seller_features = pd.DataFrame()
seller_features["seller_id"] = (df["seller_id"].unique())
seller_features = seller_features.set_index("seller_id")

In [59]:
# ====== 1. Seller Performance Score ==========

seller_summary = (
    df.groupby("seller_id")
      .agg(
          Orders=("order_id", "nunique"),
          Revenue=("payment_value", "sum"),
          Rating=("review_score", "mean")
      )
)


seller_summary["Seller_Performance_Score"] = (
    seller_summary["Orders"].rank(pct=True) * 0.4 +
    seller_summary["Revenue"].rank(pct=True) * 0.4 +
    seller_summary["Rating"].rank(pct=True) * 0.2
)

seller_features["Seller_Performance_Score"] = (
    seller_summary["Seller_Performance_Score"]
)

# ======= 2. Average Delivery Time ==========

# Actual delivery time
df["delivery_days"] = (
    df["order_delivered_customer_date"] -
    df["order_purchase_timestamp"]
).dt.days

# Delivery delay
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"] -
    df["order_estimated_delivery_date"]
).dt.days

# On-time delivery flag
df["On_Time"] = (
    df["delivery_delay_days"] <= 0
).astype(int)

# Seller Features
seller_features["Average_Delivery_Time"] = (
    df.groupby("seller_id")["delivery_days"].mean()
)

# ======= 3.Average Seller Rating =========
seller_features["Average_Review_Score"] = (df.groupby("seller_id")["review_score"].mean())

seller_features.head()

,Seller_Performance_Score,Average_Delivery_Time,Average_Review_Score
seller_id,,,
7c67e1448b00f6e969d365cea6b010ab,0.835212,22.041265,3.394773
b8bc237ba3788b23da09c0f1f3a3288c,0.833737,11.500000,4.361702
4a3ca9315b744ce9f8e9374361493884,0.856247,13.800095,3.801692
8f2ce03f928b567e3d56181ae20ae952,0.858817,13.959538,4.057143
9f505651f4a6abe901a56cdc21508025,0.893879,8.400871,4.250545


# Missing Values

In [63]:
# ======== Check Missing Values  =============

print(seller_features.info())
print(seller_features.isnull().sum())

<class 'pandas.DataFrame'>
Index: 3096 entries, 7c67e1448b00f6e969d365cea6b010ab to 51a04a8a6bdcb23deccc82b0b80742cf
Data columns (total 3 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Seller_Performance_Score  3096 non-null   float64
 1   Average_Delivery_Time     3096 non-null   float64
 2   Average_Review_Score      3096 non-null   float64
dtypes: float64(3)
memory usage: 193.8 KB
None
Seller_Performance_Score    0
Average_Delivery_Time       0
Average_Review_Score        0
dtype: int64


In [62]:
# ======== Handle Missing Values ===========

seller_features["Seller_Performance_Score"] = (
    seller_features["Seller_Performance_Score"]
    .fillna(seller_features["Seller_Performance_Score"].median())
)

seller_features["Average_Delivery_Time"] = (
    seller_features["Average_Delivery_Time"]
    .fillna(seller_features["Average_Delivery_Time"].median())
)

seller_features["Average_Review_Score"] = (
    seller_features["Average_Review_Score"]
    .fillna(seller_features["Average_Review_Score"].median())
)

In [64]:
# ========= Save Engineered Features =================

# Output directory
output_dir = r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed"

# Output file path
output_path = os.path.join(output_dir, "seller_features.csv")

# Reset index before saving
seller_features.reset_index(inplace=True)

# Save CSV
seller_features.to_csv(output_path, index=False)

print(f"Product feature dataset saved successfully!")
print(f"Location: {output_path}")

Product feature dataset saved successfully!
Location: C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed\seller_features.csv


# Session 4 - Time-Based Feature Engineering

In [65]:
# ====== 1. Purchase Month =========
df["Purchase_Month"] = (df["order_purchase_timestamp"].dt.month)

# ====== 2. Purchase Weekday =======
df["Purchase_Weekday"] = (df["order_purchase_timestamp"].dt.day_name())

# ====== 3. Purchase Hour =========
df["Purchase_Hour"] = (df["order_purchase_timestamp"].dt.hour)

# ======= 4. Season =======
def get_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    else:
        return "Spring"
df["Season"] = df["Purchase_Month"].apply(get_season)

# ======== Festival / Peak Sales Indicator ==========
peak_months = [11, 12]  # Black Friday & Christmas
df["Festival_Sales_Indicator"] = ( df["Purchase_Month"].isin(peak_months).astype(int))


df.to_csv(
    r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed\customer_analytics_featured.csv",
    index=False
)

print("Time-based features added and dataset saved successfully!")

Time-based features added and dataset saved successfully!


In [66]:
# Folder path
processed_path = r"C:\Users\RAaM\OneDrive\Desktop\E-Commerce Customer Analytics\data\processed"

# Load datasets
customer_features = pd.read_csv(os.path.join(processed_path, "customer_features.csv"))
product_features = pd.read_csv(os.path.join(processed_path, "product_features.csv"))
seller_features = pd.read_csv(os.path.join(processed_path, "seller_features.csv"))
customer_analytics_featured = pd.read_csv(os.path.join(processed_path, "customer_analytics_featured.csv"))


datasets = {
    "Customer Features": customer_features,
    "Product Features": product_features,
    "Seller Features": seller_features,
    "Featured Dataset": customer_analytics_featured
}

for name, data in datasets.items():
    print("=" * 60)
    print(name)
    print("=" * 60)
    print(data.info())
    print()

for name, data in datasets.items():
    print("\n" + "="*70)
    print(name)
    print("="*70)

    print(f"Rows    : {data.shape[0]}")
    print(f"Columns : {data.shape[1]}")
    print(f"Missing : {data.isnull().sum().sum()}")
    print(f"Duplicates : {data.duplicated().sum()}")

Customer Features
<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        96096 non-null  str    
 1   Total_Orders              96096 non-null  int64  
 2   Total_Spending            96096 non-null  float64
 3   Average_Order_Value       96096 non-null  float64
 4   Average_Basket_Size       96096 non-null  float64
 5   Recency                   96096 non-null  int64  
 6   Frequency                 96096 non-null  int64  
 7   Monetary                  96096 non-null  float64
 8   Customer_Tenure           96096 non-null  int64  
 9   Days_Since_Last_Purchase  96096 non-null  int64  
 10  Average_Review_Rating     96096 non-null  float64
 11  Preferred_Payment_Method  96096 non-null  str    
dtypes: float64(5), int64(5), str(2)
memory usage: 12.6 MB
None

Product Features
<class 'pandas.DataFrame'>
Range